In [1]:
from datasets import datasets 
for ds in datasets.values(): 
    print(ds.filelist)

/vol/ideadata/ed52egek/pycharm/syneverything/datasets/celeba.csv
/vol/ideadata/ed52egek/pycharm/syneverything/datasets/cxr-lt.csv
/vol/ideadata/ed52egek/pycharm/syneverything/datasets/mimic.csv
/vol/ideadata/ed52egek/pycharm/syneverything/datasets/imagenet_lt.csv
/vol/ideadata/ed52egek/pycharm/syneverything/datasets/isic_clean.csv
/vol/ideadata/ed52egek/pycharm/syneverything/datasets/ctrate_slices_final.csv


In [2]:
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

def make_cv_splits(
    df: pd.DataFrame,
    group_col: str = "id",
    n_splits: int = 10,
    val_size: float = 0.10,
    seed: int = 42,
):
    # safety check
    n_groups = df[group_col].nunique()
    if n_groups < n_splits:
        raise ValueError(f"Need at least {n_splits} unique {group_col}s, got {n_groups}.")

    gkf = GroupKFold(n_splits=n_splits)
    X = df.index.values
    groups = df[group_col].values

    fold_dfs = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
        # start from a copy and set everything to TRAIN
        fold_df = df.copy()
        fold_df["Split"] = "TRAIN"

        # mark TEST
        fold_df.loc[test_idx, "Split"] = "TEST"

        # now carve out VAL from the remaining TRAIN, by groups
        train_mask = fold_df["Split"] == "TRAIN"
        train_idx_only = fold_df.index[train_mask].values
        train_groups = fold_df.loc[train_idx_only, group_col].values

        gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed + fold)
        tr_sub_idx, val_sub_idx = next(gss.split(train_idx_only, groups=train_groups))

        val_indices = train_idx_only[val_sub_idx]
        fold_df.loc[val_indices, "Split"] = "VAL"

        # collect per-fold df
        fold_df["fold"] = fold
        fold_dfs.append(fold_df)

        # optional: quick sanity print
        # print(f"Fold {fold}:",
        #       fold_df["Split"].value_counts().to_dict())

    return fold_dfs  # list of length n_splits

# Usage:
# Example: save each fold
# for f, d in enumerate(fold_dfs):
#     d.to_csv(f"cv_fold_{f}.csv", index=False)

#dff = fold_dfs[0]
#dff[dff.id == 8][["fold", "Split", "id"]]

In [3]:
import pandas as pd
from sklearn.model_selection import GroupKFold
import os 



for ds_name, dataset in datasets.items(): 
    df = pd.read_csv(dataset.filelist)
    df = df[df.Split == "TRAIN"].reset_index(drop=True)
    df = df.drop(columns={"Unnamed: 0"}, errors="ignore")

    gkf = GroupKFold(n_splits=5)

    # Assign a fold index to each row
    df["fold"] = -1
    for fold, (train_idx, val_idx) in enumerate(gkf.split(df.index, None, df["id"])):
        df.loc[val_idx, "fold"] = fold


    fold_dfs = make_cv_splits(df, group_col="id", n_splits=5, val_size=0.10, seed=42)
    for i, fold_df in enumerate(fold_dfs): 
        path = f"/vol/ideadata/ed52egek/pycharm/syneverything/datasets/kfold/{ds_name}/"
        os.makedirs(path , exist_ok=True)
        fold_df.to_csv(path+ f"fold_{i}.csv")

In [4]:
fold_df["2h"].sum()

np.int64(147)

In [5]:
df = pd.read_csv(datasets["mimic"].filelist)



In [6]:
df.sum()

id                                                                   2994638885
path                          processed_files/files/p19/p19999987/s58971208/...
Split                         TRAINTRAINTRAINTRAINTRAINTRAINTRAINTRAINTRAINT...
Atelectasis                                                             12168.0
Cardiomegaly                                                            12719.0
Consolidation                                                            2436.0
Edema                                                                    6777.0
Enlarged Cardiomediastinum                                               1685.0
Fracture                                                                  969.0
Lung Lesion                                                              1525.0
Lung Opacity                                                            13721.0
No Finding                                                              69677.0
Pleural Effusion                        